# School Absence and Parental Crime
## Running Analysis

This notebook walks you through the creation of the results created in `/scripts/run_analysis.py` step-by-step, incl. the [appendix](#Appendix).

For this notebook to work, make sure the project is installed *including optional dependencies*:

In the terminal:
```
pip install -e .[notebooks]
```

In [30]:
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

from absence_and_crime.plotting import period_mean, twfe, coef_plot

from absence_and_crime.config import PATHS, setup_matplotlib
paths = PATHS
setup_matplotlib()

In [2]:
covariates = pd.read_parquet(paths.temp / "covariates.parquet")
families = pd.read_parquet(paths.temp / "families.parquet")
absence = pd.read_parquet(paths.temp / "absence.parquet")

with open(paths.temp / "panels.pkl", "rb") as f:
    panels = pickle.load(f)

with open(paths.temp / "instability.pkl", "rb") as f:
    instability = pickle.load(f)

with open(paths.temp / "records.pkl", "rb") as f:
    records = pickle.load(f)

#### Getting to N

In [ ]:
in_absence = families.pnr.nunique()
parental_crime = (
    families
    .merge(
        records["crime"].rename(columns={"pnr": "parent"}), how="inner"
    )
    .loc[
        lambda d: (d.month >= d.start_of_school) & (d.month.dt.year.between(d.parent_start, d.parent_end))
    ].pnr.nunique()
)
analytical_sample = panels["crime"].pnr.nunique()

print(f'''
    We find {in_absence:,.0f} children in the absence registers with at least one registered parent.
    Of these, {parental_crime:,.0f} have a registered parental crime since starting school.
    Of these, {analytical_sample:,.0f} experience the parental crime while being in the absence register. 
''')

### Table 1: Sample descriptives by parent crime and incarceration

In [4]:
df = (
    covariates
    .merge( # add grade at time of crime
        panels["crime"]
        .loc[
            lambda d: d.event_time.le(0), 
            ["pnr", "event_time", "grade"]
            ]
        .sort_values(["pnr", "event_time"])
        .groupby("pnr", as_index=False).last()[["pnr", "grade"]],
        on=["pnr"], how="left", validate="1:1"
    )
    .merge( # add absence rate
        absence
        .groupby("pnr")["abs_rate"]
        .mean().reset_index(),
        on="pnr", how="left", validate="1:1"
    )
    .assign( 
        crime = lambda d: d.pnr.isin(set(panels["crime"].pnr)),
        incar = lambda d: d.pnr.isin(set(panels["incarceration"].pnr))
    )
)

In [5]:
# experienced crime in grades K-2
df["k_to_2"] = df.grade.le(2).where(df.grade.notna())

# college degree or higher
df["college"] = df.education.eq(4).where(df.education.notna())

In [ ]:
vl = ["female", "k_to_2", "non_danish", "wages", "assets", "fulltime", "college", "abs_rate"]
var_names = {
    "female": "Female",
    "k_to_2": "Grade K-2 at crime",
    "non_danish": "1st or 2nd gen immigrant",
    "wages": "Earnings",
    "assets": "Net assets",
    "fulltime": "F-T employment",
    "college": "College or higher",
    "abs_rate": "Absence rate"
}
stats = ["mean", "std"]
groups = ["Crime", "Incarceration", "None"]
conds = [df.crime, df.incar, ~df.crime]

tbl = []
for c, g in zip(conds, groups):
    block = df.loc[c, vl].agg(stats).transpose().round(2)
    block.columns = pd.MultiIndex.from_product([[g], block.columns])
    nobs = pd.DataFrame([[len(df[c])] + [None]*(len(stats)-1)], columns=block.columns, index=["N"])
    block = pd.concat([block, nobs], axis=0)
    tbl.append(block)

tbl = pd.concat(tbl, axis=1)
tbl = tbl.rename(index=var_names)

tbl.to_html(paths.tables / "Table 1.html")
display(tbl)

### Figure 1: Descriptive time trends of school absence rate relative to parental crime

In [ ]:
fig, ax = period_mean(panels["crime"])
fig.savefig(paths.figures / "Figure 1.png")

### Figure 2: Event-study regression results of parent crime predicting school absence

In [ ]:
# Run regression
res, coefs = twfe(panels["crime"])

# Plot
fig, ax = coef_plot(coefs, event="first parent crime")
fig.savefig(paths.figures / "Figure 2.png")

# Test pre-trend
print(
    'Joint significane of period dummies prior to t=-1: ',
    f'p={coefs.attrs["test_periods"]["pval"]:.2f}'
    )

# Show estimates for t=0 to t=6
display(coefs.set_index("period").loc[0:6].round(3).T)

### Figure 3: Descriptive timing of parent and household instability relative to parent crime

In [9]:
# Set up data
df = panels["crime"].copy()

id_shift = {"pnr": "parent"}  # for parent indicators

for k, v in instability.items():
    df = df.merge(
        v.rename(columns=id_shift if k not in ["address", "transfer"] else {}),
        on=(
            ["parent", "month"]
            if k not in ["address", "transfer"]
            else ["pnr", "month"]
        ),
        how="left",
        validate="m:1",
        indicator=k,
    )
    # has registered instability (x100 to get from shares to pct.)
    df[k] = df[k].eq("both").astype(int) * 100

In [ ]:
# Plot
fig, axes = plt.subplots(
    2, 2, figsize=(16, 12), sharex=True, constrained_layout=True
)
axes = axes.ravel()

indicator_names = {
    "address": "Child address change",
    "mental": "Parent psych. diagnosis",
    "jobloss": "Parent job loss",
    "substance": "Parent substance abuse",
}

for i, k in enumerate(instability):
    period_mean(
        df, y=k, ax=axes[i], ylim=None, ytitle=indicator_names[k] + " (pct.)"
    )
    if i < 2:
        axes[i].set_xlabel("")

fig.savefig(paths.figures / "Figure 3.png")

### Figure 4: Stratified by dimensions of parent and household instability

In [ ]:
# Reuse data from Figure 3...

# Plot
fig, axes = plt.subplots(
    2, 2, figsize=(16, 12), constrained_layout=True, sharey=True, sharex=True
)
axes = axes.ravel()

for i, k in enumerate(instability):

    # Split by household instability
    df["after"] = df[k].astype(bool) & df["event_time"].between(0, 3)
    df["after"] = (
        df.groupby("pnr")["after"].transform(lambda g: (g == 1).any()).astype(bool)
    )

    # Plot
    period_mean(
        df[df.after],
        lfit=False,
        label="Yes",
        color=".4",
        ax=axes[i],
        plot_kwargs={"marker": "^", "markersize": 8},
    )

    period_mean(
        df[~df.after],
        lfit=False,
        label="No",
        event="first parent crime",
        vline=None,
        ax=axes[i],
        plot_kwargs={"markersize": 8},
    )

    # Edits
    handles, labels = axes[i].get_legend_handles_labels()
    leg = axes[i].legend(
        handles,
        labels,
        loc="upper left",
        ncol=2,
        frameon=False,
        title=indicator_names[k] + ":",
    )
    leg.get_title().set_fontweight("bold")
    axes[i].set_ylim(6, 12)
    if i % 2 != 0:
        axes[i].set_ylabel("")
    if i < 2:
        axes[i].set_xlabel("")


fig.savefig(paths.figures / "Figure 4.png")


### Figure 5: Descriptive time trends of school absence rate relative to parent arrest, charge, conviction, and incarceration

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12), sharey=True, constrained_layout=True)
axes = axes.ravel()

events = ["arrest", "charge", "conviction", "incarceration"]

for i, k in enumerate(events):
    period_mean(panels[k], ax=axes[i], event=f"first parent {k}")
    if i % 2 != 0:
        axes[i].set_ylabel("")

fig.savefig(paths.figures / "Figure 5.png")

### Figure 6: Event-study regression results around parent arrest, charge, conviction and incarceration

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12), sharey=True, constrained_layout=True)
axes = axes.ravel()

for i, k in enumerate(events):
    res, coefs = twfe(panels[k])
    coef_plot(coefs, ax=axes[i], event=f"first parent {k}", ylim=(-1, 1.5))
    
    # Show estimates and test statistics for arrest results
    if k=="arrest":
        display(coefs.set_index("period").loc[0:6].round(3).T)
        print(
            'Joint significane of period dummies prior to t=-1: ',
            f'p={coefs.attrs["test_periods"]["pval"]:.2f}'
            )

    if i % 2 != 0:
        axes[i].set_ylabel("")

fig.savefig(paths.figures / "Figure 6.png")

## Appendix

### Figure A1: Timing of Parental Criminal Justice Contact Following Parental Crime

In [14]:
# ---- Set up data

df = panels["crime"].loc[lambda d: d.event_time.between(0, 12)].copy()

contact_steps = ["arrest", "charge", "conviction", "incarceration"]
contact_names = ["Arrested", "Charged", "Convicted", "Incarcerated"]

# Indicate criminal record in month t
for k in contact_steps:
    df = df.merge(
        records[k].rename(columns={"pnr": "parent"}),
        on=["parent", "month"],
        how="left",
        validate="m:1",
        indicator=k,
    )
    df[k] = df[k].eq("both")

# Period means
agg_dict = {k: (k, "mean") for k in contact_steps}
agg_dict["N"] = ("pnr", "size")
df = df.groupby("event_time", as_index=False).agg(**agg_dict)

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
colors = ["0", ".2", ".4", ".6"]
styles = ["-", "--", "-.", ":"]

for i, k in enumerate(contact_steps):
    # discretion
    ones = df[k] * df["N"]
    zeros = (1-df[k]) * df["N"]
    df.loc[ones <= 3, k] = 0
    df.loc[zeros <= 3, k] = 1

    # plot
    ax.plot(
        df.event_time, 
        df[k], 
        linestyle=styles[i], 
        color=colors[i], 
        label=contact_names[i]
    )

# Edit
ax.legend(bbox_to_anchor=(0.5, -0.12), ncol=2, title="Parent:")

ax.set_xlabel("Months since first parent crime")
ax.set_ylabel("Share of children")
ax.tick_params(axis="y", rotation=90)
ax.set_yticks(np.arange(0, 5, 1) / 10)
ax.yaxis.set_major_formatter(
    FuncFormatter(
        lambda x, _: "0" if abs(x) < 1e-12 else f"{x:.2f}".replace("0.", ".")
    )
)

fig.savefig(paths.figures / "Figure A1.png")

In [ ]:
print(
    f'{df.loc[df.event_time == 0, "arrest"].sum():.0%} of children experience a parental arrest in the month of the first parent crime.\n',
    f'{df.loc[df.event_time <= 3, "arrest"].sum():.0%} of children experience a parental arrest within 3 months of the first parent crime.'
)

In [17]:
denominator = panels["arrest"].pnr.nunique() # number of children with parental arrest

df = (
    panels["arrest"][["pnr", "parent", "t0"]].drop_duplicates()
    .merge(
        records["crime"].rename(columns={"pnr": "parent"}), 
        on=["parent"], how="inner"
    )
    .assign(relative_month = lambda d: d.month.astype("int64") - d.t0.astype("int64"))
    .loc[lambda d: d.relative_month.between(-60, 0)]
    .groupby("pnr").relative_month.max().reset_index()
)

df["relative_month"] = df["relative_month"].clip(lower=-12)

numerator = df.groupby("relative_month").pnr.count()
s = numerator / denominator

In [ ]:
print(
    f'{s.loc[0].sum():.0%} of children experience a parental crime within the month of parental arrest.\n',
    f'{s.loc[range(-3, 0 + 1)].sum():.0%} of children experience a parental crime within 3 months prior to parental arrest.\n',
)

### Figure A2: Regression results stratified by dimensions of parent and household instability

In [19]:
# Set up data
df = panels["crime"].copy()

id_shift = {"pnr": "parent"}  # for parent indicators

for k, v in instability.items():
    df = df.merge(
        v.rename(columns=id_shift if k not in ["address", "transfer"] else {}),
        on=(
            ["parent", "month"]
            if k not in ["address", "transfer"]
            else ["pnr", "month"]
        ),
        how="left",
        validate="m:1",
        indicator=k,
    )
    # has registered instability (x100 to get from shares to pct.)
    df[k] = df[k].eq("both").astype(int) * 100

In [ ]:
# Plot
fig, axes = plt.subplots(
    2, 2, figsize=(16, 12), constrained_layout=True, sharey=True, sharex=True
)
axes = axes.ravel()

for i, k in enumerate(instability):

    # ---- Setup data

    # Split by household instability
    df["after"] = df[k].astype(bool) & df["event_time"].between(0, 3)
    df["after"] = (
        df.groupby("pnr")["after"].transform(lambda g: (g == 1).any()).astype(bool)
    )

    # Plot
    _, coefs = twfe(df[df.after])
    coef_plot(
        coefs,
        ax=axes[i],
        label="Yes",
        color=".4",
        errorbar_kwargs={"markersize": 8},
    )

    _, coefs = twfe(df[~df.after])
    coef_plot(
        coefs,
        event="first parent crime",
        ylim=None,
        ax=axes[i],
        label="No",
        vline=None,
        errorbar_kwargs={"markersize": 8},
    )

    # Edits
    handles, labels = axes[i].get_legend_handles_labels()
    leg = axes[i].legend(
        handles,
        labels,
        loc="upper left",
        ncol=2,
        frameon=False,
        title=indicator_names[k],
    )
    leg.get_title().set_fontweight("bold")
    axes[i].set_ylim((-3, 3.5))
    if i % 2 != 0:
        axes[i].set_ylabel("")
    if i < 2:
        axes[i].set_xlabel("")

fig.savefig(paths.figures / "Figure A2.png")

### Figure A3: Separating Crime and Arrest

In [21]:
delta = 3 # required distance in months

In [22]:
# ---- Set up data

def separate_events(
    df: pd.DataFrame,
    exclusion_events: pd.DataFrame,
    *,
    id_col: str = "parent",
    t0_col: str = "t0",
    month_col: str = "month",
    delta: int = 1,
):
    """
    Remove individuals from event panel who have another event 
    occuring within `delta` months.

    Parameters
    ----------
    df
        Event panel DataFrame.
    exclusion_events
        DataFrame of events to base exclusion on (id, calendar_month_col).
    id_col
        Name of the id column in `df` and `exclusion_events` (these have to be the same).
    t0_col
        Name of column in `df` containing month of event (must be dtype Period[M]).
    month_col 
        Name of column in `exclusion_events` containing month of event 
        (must be dtype Period[M]).
    delta
        Minimum required distance in months between events.
        E.g., delta = 0 -> exclusion events may not occur in t0 itself.

    Returns
    -------
    df_filtered
        Filtered panel.
    """

    # unique (id, t0) pairs
    id_t0 = df[[id_col, t0_col]].drop_duplicates()

    # merge on exclusion events 
    exclude = (
        id_t0
        .merge(exclusion_events, on=id_col, how="left")
        .loc[
            lambda d: (
                d[month_col].astype("int64") - d[t0_col].astype("int64")
                ).abs() <= delta, 
            [id_col, t0_col]
        ]
        .drop_duplicates()
    )

    df_filtered = df.merge(
        exclude, on=[id_col, t0_col], how="left", validate="m:1", indicator=True
        ).loc[lambda d: d._merge.eq("left_only")].drop(columns=["_merge"])
    
    return df_filtered

In [23]:
df = separate_events(panels["crime"], records["arrest"].rename(columns={"pnr": "parent"}), delta=delta)

In [ ]:
print(
    f'Of the {panels["crime"].pnr.nunique():,.0f} children experiencing a parental crime, {df.pnr.nunique():,.0f}',
     f'({df.pnr.nunique()/panels["crime"].pnr.nunique():.0%}) do not have a parental arrest within {delta} months.',
    )

In [ ]:
# Run regression
res, coefs = twfe(df)

# Plot
fig, ax = coef_plot(coefs, event="first parent crime")
fig.savefig(paths.figures / "Figure A3.png")

# Test pre-trend
print(
    'Joint significane of period dummies prior to t=-1: ',
    f'p={coefs.attrs["test_periods"]["pval"]:.2f}'
    )

# Show estimates for t=0 to t=6
display(coefs.set_index("period").loc[0:6].round(3).T)

### Figure A4: Only not-yet treated

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), dpi=80, constrained_layout=True)

# Run regression
_, coefs = twfe(
    panels["crime"].loc[lambda d: d.event_time.lt(0)],
    window=range(-12, 0), 
)
_, coefs_main = twfe(panels["crime"])
coefs_main = coefs_main.loc[lambda d: d.period.between(-12, -1)]

# Plot
coef_plot(coefs, color="0", ecolor="0", ax=ax, label="Not-yet-treated children")
coef_plot(
    coefs_main, 
    event="first parent crime",
    color=".4",
    ecolor=".4", 
    xjump=2, 
    ax=ax, 
    label = "All children"
)
ax.legend(loc="upper left")
fig.savefig(paths.figures / "Figure A4.png")

# Test pre-trend
print(
    'Joint significane of period dummies prior to t=-1: ',
    f'p={coefs.attrs["test_periods"]["pval"]:.2f}'
    )